In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32)



In [ ]:
# 2. Create TensorDataset objects
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)


In [ ]:
# 3. Create DataLoaders
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)




In [ ]:
# 4. Print shape of one batch
# Get one batch
X_batch, y_batch = next(iter(train_loader))

print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)


In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt
X_batch, y_batch = next(iter(train_loader))

fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for i in range(5):
    image = X_batch[i]
    if image.ndim == 3:
        image = image.permute(1, 2, 0)

    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(f"Label: {y_batch[i].item()}")
    axes[i].axis('off')

plt.show()


In [ ]:
# Task 1: Write your model class here:
import torch
import torch.nn as nn

class NeuralNet(nn.Module):
    def __init__(self, input_size):
        super(NeuralNet, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        return self.model(x)


In [ ]:
# Task 2: Write your training loop here:
def train_loop(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        predictions = model(X).squeeze()
        loss = loss_fn(predictions, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
# Task 3: Write your validation loop here:
def val_loop(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            predictions = model(X).squeeze()
            loss = loss_fn(predictions, y)
            total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
# Task 4: Define device, model, loss, optimizer:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sample_X, _ = next(iter(train_loader))
input_size = sample_X.view(sample_X.size(0), -1).shape[1]

model = NeuralNet(input_size).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_loop(model, train_loader, loss_fn, optimizer, device)
    val_loss = val_loop(model, test_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
model.eval()
X_batch, y_batch = next(iter(test_loader))
X_batch, y_batch = X_batch.to(device), y_batch.to(device)
with torch.no_grad():
    predictions = model(X_batch).squeeze()
# Move data to CPU for plotting
X_batch = X_batch.cpu()
y_batch = y_batch.cpu()
predictions = predictions.cpu()

fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for i in range(5):
    image = X_batch[i]
    if image.ndim == 3:
        image = image.permute(1, 2, 0)

    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(
        f"Pred: {predictions[i].item():.1f}\nActual: {y_batch[i].item():.1f}"
    )
    axes[i].axis('off')

plt.show()
